# 05 — Modeling: Engine Rekomendasi Hybrid (prototipe `06_api/recommend.py`)

```
score = 0.30*similarity_konten + 0.35*kedekatan_geo
      + 0.20*popularitas + 0.15*buka_di_hari_terpilih
```

Cold-start friendly — tidak butuh riwayat user.
Baca `merged_venues_enriched.csv` (219 venue final).

> **Catatan**: ini prototipe scoring hybrid di `06_api/`, BUKAN modeling GA/PSO
> final. Output di bawah perlu di-run ulang terhadap dataset 219 venue terkini.

In [ ]:
import sys
import os
sys.path.insert(0, "../..")
sys.path.insert(0, "../../06_api")

import pandas as pd
import matplotlib.pyplot as plt
from recommend import Recommender, WEIGHTS, DAYS

## Bobot komponen skor

In [ ]:
pd.Series(WEIGHTS).plot(kind="bar", title="Bobot komponen scoring")
plt.ylabel("bobot")
plt.show()

## Inisialisasi engine

In [ ]:
import os
csv_path = os.path.join(os.path.dirname(os.path.abspath(".")), 
                        "data", "processed", "merged_venues_enriched.csv")
# Dari docs/notebooks/ -> naik 2 level ke root
csv_path = "../../data/processed/merged_venues_enriched.csv"

rec = Recommender(csv_path=csv_path)
print(f"Total venue ter-load: {len(rec.df)}")
print(f"Kolom: {list(rec.df.columns[:10])}...")

## Demo: turis di Monas, suka museum, kunjungan Sabtu

In [ ]:
res = rec.recommend(lat=-6.1754, lon=106.8272,
                     category="Museum", day="Sabtu", top_n=10)
res

## Demo: cold-start, tanpa preferensi kategori (netral)

In [ ]:
res_netral = rec.recommend(lat=-6.1754, lon=106.8272, top_n=10)
res_netral

## Pengaruh jarak terhadap skor (sanity check komponen geo)

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(res["distance_km"], res["score"], alpha=0.7)
plt.xlabel("distance_km")
plt.ylabel("score")
plt.title("Skor vs jarak (kategori museum, Sabtu)")
plt.show()

## Eksperimen: ubah filter `only_open` (venue tutup di hari terpilih di-exclude)

In [ ]:
res_open_only = rec.recommend(lat=-6.1754, lon=106.8272,
                               category="Museum", day="Sabtu",
                               top_n=10, only_open=True)
print(f"Tanpa filter only_open: {len(res)} hasil")
print(f"Dengan filter only_open: {len(res_open_only)} hasil")